In [ ]:
# ============================================================
# IMPORTS — run this first, every session
# ============================================================
import json
import time
import random
import inspect
import os
import numpy as np
import torch
import json, time, random
import numpy as np

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print("✅ Imports done.")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
✅ Imports done.


In [ ]:
# ============================================================
# CORE FUNCTIONS — run once per session
# ============================================================

# ── Feasibility ──────────────────────────────────────────────
def is_feasible(sol, weights, capacities, n, m):
    for i in range(m):
        total = sum(weights[i][j] * sol[j] for j in range(n))
        if total > capacities[i]:
            return False
    return True

# ── Fitness (raw value, no penalty) ─────────────────────────
def fitness(sol, values, n):
    return sum(values[j] * sol[j] for j in range(n))

# ── Efficiency scores for repair ────────────────────────────
def compute_efficiency_scores(values, weights, capacities, n, m):
    scores = []
    for j in range(n):
        denom = sum(weights[i][j] / capacities[i] for i in range(m))
        scores.append(values[j] / denom if denom > 0 else 0)
    return scores

# ── Repair operator ──────────────────────────────────────────
def repair(sol, values, weights, capacities, efficiency_scores, n, m):
    sol = list(sol).copy()
    order = sorted(range(n), key=lambda j: efficiency_scores[j])
    # drop lowest efficiency items until feasible
    for j in order:
        if is_feasible(sol, weights, capacities, n, m):
            break
        sol[j] = 0
    # greedily add highest efficiency items if room
    for j in reversed(order):
        if sol[j] == 0:
            sol[j] = 1
            if not is_feasible(sol, weights, capacities, n, m):
                sol[j] = 0
    return sol

# ── Initialization ───────────────────────────────────────────
def initialize_population(population_size, n):
    return [[random.randint(0, 1) for _ in range(n)]
            for _ in range(population_size)]

def initialize_population_biased(population_size, n, p_j_values):
    return [[1 if random.random() < p_j_values[j] else 0
             for j in range(n)]
            for _ in range(population_size)]

# ── Tournament selection ─────────────────────────────────────
def tournament_select(population, fitness_scores):
    a = random.randint(0, len(population) - 1)
    b = random.randint(0, len(population) - 1)
    if fitness_scores[a] >= fitness_scores[b]:
        return population[a]
    else:
        return population[b]

# ── Uniform crossover ────────────────────────────────────────
def uniform_crossover(parent1, parent2, n):
    return [parent1[j] if random.random() < 0.5 else parent2[j]
            for j in range(n)]

# ── Mutation ─────────────────────────────────────────────────
def mutate(sol, n, rate=None):
    rate = rate if rate is not None else 1 / n
    return [1 - sol[j] if random.random() < rate else sol[j]
            for j in range(n)]

# ── Quick sanity check ───────────────────────────────────────
def sanity_check(instances):
    for alpha, inst in instances.items():
        w = np.array(inst['weights'])
        assert w.shape == (M, N), f"Wrong shape for α={alpha}: {w.shape}"
        print(f"  α={alpha}: weights shape={w.shape} ✅")

print("Core functions defined.")

Core functions defined.


In [ ]:
# ============================================================
# GA FUNCTIONS — run once per session
# ============================================================

# ── Repair-based GA (used for reference best finder) ─────────
def run_ga(n, m, values, weights, capacities,
           population_size=50,
           max_offspring=10000,
           best_known=None,
           target_pct=0.99,
           init_method='uniform',
           p_j_values=None,
           seed=None):

    if seed is not None:
        random.seed(seed)

    efficiency_scores = compute_efficiency_scores(
        values, weights, capacities, n, m
    )

    if init_method == 'uniform':
        population = initialize_population(population_size, n)
    else:
        population = initialize_population_biased(
            population_size, n, p_j_values
        )

    feasible_before_repair = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    for idx in range(population_size):
        population[idx] = repair(
            population[idx], values, weights,
            capacities, efficiency_scores, n, m
        )

    fitness_scores   = [fitness(sol, values, n) for sol in population]
    avg_fitness_gen0 = sum(fitness_scores) / population_size
    best_fitness     = max(fitness_scores)
    best_solution    = population[fitness_scores.index(best_fitness)].copy()

    convergence_point = None
    if best_known and best_fitness >= best_known * target_pct:
        convergence_point = 0

    offspring_count = 0
    while offspring_count < max_offspring:
        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)
        child   = uniform_crossover(parent1, parent2, n)
        child   = mutate(child, n)
        child   = repair(child, values, weights,
                         capacities, efficiency_scores, n, m)

        child_fitness = fitness(child, values, n)
        worst_idx     = fitness_scores.index(min(fitness_scores))

        if child_fitness > fitness_scores[worst_idx]:
            population[worst_idx]     = child
            fitness_scores[worst_idx] = child_fitness

        if child_fitness > best_fitness:
            best_fitness  = child_fitness
            best_solution = child.copy()

        offspring_count += 1

        if (convergence_point is None and best_known and
                best_fitness >= best_known * target_pct):
            convergence_point = offspring_count

    return {
        'feasibility_rate_gen0': feasible_before_repair,
        'avg_fitness_gen0':      avg_fitness_gen0,
        'best_fitness':          best_fitness,
        'convergence_point':     convergence_point,
        'gap_pct': ((best_known - best_fitness) / best_known * 100
                    if best_known else None)
    }


# ── Death penalty GA ─────────────────────────────────────────
def run_ga_death_penalty(n, m, values, weights, capacities,
                          population_size=50,
                          max_offspring=20000,
                          best_known=None,
                          target_pct=0.99,
                          init_method='uniform',
                          p_j_values=None,
                          seed=None,
                          track_every=50):

    if seed is not None:
        random.seed(seed)

    if init_method == 'uniform':
        population = initialize_population(population_size, n)
    else:
        population = initialize_population_biased(
            population_size, n, p_j_values
        )

    feasible_before = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    def death_fitness(sol):
        if not is_feasible(sol, weights, capacities, n, m):
            return 0
        return sum(values[j] * sol[j] for j in range(n))

    fitness_scores   = [death_fitness(sol) for sol in population]
    avg_fitness_gen0 = sum(fitness_scores) / population_size
    best_fitness     = max(fitness_scores)
    best_solution    = population[fitness_scores.index(best_fitness)].copy()
    pct_of_best_gen0 = (best_fitness / best_known * 100
                        if best_known and best_fitness > 0 else 0.0)

    convergence      = {t: None for t in CONVERGENCE_THRESHOLDS}
    curve            = [(0, best_fitness, avg_fitness_gen0)]
    offspring_count  = 0

    while offspring_count < max_offspring:
        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)
        child   = uniform_crossover(parent1, parent2, n)
        child   = mutate(child, n)

        child_fitness = death_fitness(child)
        worst_idx     = fitness_scores.index(min(fitness_scores))

        if child_fitness > fitness_scores[worst_idx]:
            population[worst_idx]     = child
            fitness_scores[worst_idx] = child_fitness

        if child_fitness > best_fitness:
            best_fitness  = child_fitness
            best_solution = child.copy()

        offspring_count += 1

        if best_known and is_feasible(best_solution, weights, capacities, n, m):
            for t in CONVERGENCE_THRESHOLDS:
                if convergence[t] is None and best_fitness >= best_known * (t/100):
                    convergence[t] = offspring_count

        if offspring_count % track_every == 0:
            curve.append((offspring_count, best_fitness,
                          sum(fitness_scores) / population_size))

    best_feasible = is_feasible(best_solution, weights, capacities, n, m)

    return {
        'feasibility_rate_gen0':  feasible_before,
        'avg_fitness_gen0':       avg_fitness_gen0,
        'pct_of_best_gen0':       pct_of_best_gen0,
        'best_fitness_final':     best_fitness,
        'best_solution_feasible': best_feasible,
        'pct_of_best_final':      (best_fitness / best_known * 100
                                   if best_known and best_feasible else 0.0),
        'convergence':            convergence,
        'curve':                  curve
    }


# ── Soft penalty GA ──────────────────────────────────────────
def run_ga_penalty_fitness(n, m, values, weights, capacities,
                            population_size=50,
                            max_offspring=20000,
                            best_known=None,
                            target_pct=0.99,
                            init_method='uniform',
                            p_j_values=None,
                            penalty_multiplier=5.0,
                            seed=None,
                            track_every=50):

    if seed is not None:
        random.seed(seed)

    if init_method == 'uniform':
        population = initialize_population(population_size, n)
    else:
        population = initialize_population_biased(
            population_size, n, p_j_values
        )

    feasible_before = sum(
        1 for sol in population
        if is_feasible(sol, weights, capacities, n, m)
    ) / population_size

    def pfitness(sol):
        total_value = sum(values[j] * sol[j] for j in range(n))
        penalty = 0
        for i in range(m):
            total_weight = sum(weights[i][j] * sol[j] for j in range(n))
            violation    = max(0, total_weight - capacities[i])
            penalty     += penalty_multiplier * violation
        return total_value - penalty

    fitness_scores   = [pfitness(sol) for sol in population]
    avg_fitness_gen0 = sum(fitness_scores) / population_size
    best_fitness     = max(fitness_scores)
    best_solution    = population[fitness_scores.index(best_fitness)].copy()
    pct_of_best_gen0 = (best_fitness / best_known * 100
                        if best_known and best_fitness > 0 else 0.0)

    convergence      = {t: None for t in CONVERGENCE_THRESHOLDS}
    curve            = [(0, best_fitness, avg_fitness_gen0)]
    offspring_count  = 0

    while offspring_count < max_offspring:
        parent1 = tournament_select(population, fitness_scores)
        parent2 = tournament_select(population, fitness_scores)
        child   = uniform_crossover(parent1, parent2, n)
        child   = mutate(child, n)

        child_fitness = pfitness(child)
        worst_idx     = fitness_scores.index(min(fitness_scores))

        if child_fitness > fitness_scores[worst_idx]:
            population[worst_idx]     = child
            fitness_scores[worst_idx] = child_fitness

        if child_fitness > best_fitness:
            best_fitness  = child_fitness
            best_solution = child.copy()

        offspring_count += 1

        if best_known and is_feasible(best_solution, weights, capacities, n, m):
            for t in CONVERGENCE_THRESHOLDS:
                if convergence[t] is None and best_fitness >= best_known * (t/100):
                    convergence[t] = offspring_count

        if offspring_count % track_every == 0:
            curve.append((offspring_count, best_fitness,
                          sum(fitness_scores) / population_size))

    best_feasible = is_feasible(best_solution, weights, capacities, n, m)

    return {
        'feasibility_rate_gen0':  feasible_before,
        'avg_fitness_gen0':       avg_fitness_gen0,
        'pct_of_best_gen0':       pct_of_best_gen0,
        'best_fitness_final':     best_fitness,
        'best_solution_feasible': best_feasible,
        'pct_of_best_final':      (best_fitness / best_known * 100
                                   if best_known and best_feasible else 0.0),
        'convergence':            convergence,
        'curve':                  curve
    }

print("GA functions defined.")

GA functions defined.


In [ ]:
# ============================================================
# SUMMARY FUNCTION — run once per session
# ============================================================
def summarize_comparison(results_uni, results_bia, label, ref):

    def thresh_stats(results, t):
        vals = [r['convergence'][t] for r in results
                if r['convergence'][t] is not None]
        if not vals:
            return None, 0.0
        return np.mean(vals), len(vals) / len(results)

    def fmt(val):
        return f"{val:.0f}" if val is not None else "never"

    def improvement(u, b):
        if u is None or b is None:
            return "N/A"
        return f"{(u - b) / u * 100:+.1f}%"

    feas0_u = np.mean([r['feasibility_rate_gen0'] for r in results_uni])
    feas0_b = np.mean([r['feasibility_rate_gen0'] for r in results_bia])
    fit0_u  = np.mean([r['pct_of_best_gen0']      for r in results_uni])
    fit0_b  = np.mean([r['pct_of_best_gen0']      for r in results_bia])
    fitf_u  = np.mean([r['pct_of_best_final']     for r in results_uni])
    fitf_b  = np.mean([r['pct_of_best_final']     for r in results_bia])

    print(f"\n{'='*65}")
    print(f"  SUMMARY — {label}")
    print(f"{'='*65}")
    print(f"  Reference best : {ref}")
    print(f"  {'Metric':<45} {'Uniform':>9} {'Biased':>9}")
    print(f"  {'-'*63}")
    print(f"  {'Feasibility rate at gen 0':<45} {feas0_u:>9.3f} {feas0_b:>9.3f}")
    print(f"  {'Fitness at gen 0 (% of ref)':<45} {fit0_u:>8.1f}% {fit0_b:>8.1f}%")
    print(f"  {'Final fitness (% of ref)':<45} {fitf_u:>8.1f}% {fitf_b:>8.1f}%")
    print()

    for t in CONVERGENCE_THRESHOLDS:
        c_u, r_u = thresh_stats(results_uni, t)
        c_b, r_b = thresh_stats(results_bia, t)
        print(f"  {f'Offspring to {t}%':<45} {fmt(c_u):>9} {fmt(c_b):>9}")
        print(f"    {'improvement':<43} {improvement(c_u, c_b):>19}")

    print()
    for t in CONVERGENCE_THRESHOLDS:
        c_u, r_u = thresh_stats(results_uni, t)
        c_b, r_b = thresh_stats(results_bia, t)
        print(f"  Never reached {t}%  "
              f"Uniform={1-r_u:.0%}  Biased={1-r_b:.0%}")

    print(f"\n  {'-'*63}")
    print(f"  RESULTS LOG")
    print(f"  Experiment : {label}")
    print(f"  Reference  : {ref}")
    print(f"  Feas @ 0   : Uniform={feas0_u:.1%}  Biased={feas0_b:.1%}")
    print(f"  Fit  @ 0   : Uniform={fit0_u:.1f}%  Biased={fit0_b:.1f}%")
    for t in CONVERGENCE_THRESHOLDS:
        c_u, _ = thresh_stats(results_uni, t)
        c_b, _ = thresh_stats(results_bia, t)
        print(f"  Conv to {t}% : Uniform={fmt(c_u)}  Biased={fmt(c_b)}")
    print(f"  Final qual : Uniform={fitf_u:.1f}%  Biased={fitf_b:.1f}%")
    print(f"{'='*65}\n")

print("Summary function defined.")

Summary function defined.


In [ ]:
# ============================================================
# INSTANCE GENERATION — locked, never changes
# ============================================================
import json, time, random
import numpy as np

N, M, INSTANCE_SEED      = 100, 3, 42
ALPHA_VALUES              = [0.25, 0.50, 0.75]
POP_SIZE                  = 50
N_RUNS                    = 20
N_OFFSPRING               = 20_000
TRACK_EVERY               = 50
PENALTY_MULT              = 5.0
PJ_SAMPLES                = 10_000_000
PJ_SEED                   = 42
REF_OFFSPRING             = 30_000
REF_RUNS                  = 20
REF_POP                   = 50
CONVERGENCE_THRESHOLDS    = [50, 60, 70, 80, 90, 95, 98, 99]

def generate_random_mkp_instance(n, m, alpha, seed=42):
    np.random.seed(seed)
    values     = np.random.randint(1, 100, n)
    weights    = np.random.randint(1,  50, (m, n))
    capacities = np.array([int(alpha * weights[i].sum()) for i in range(m)])
    return values, weights, capacities

instances = {}
for alpha in ALPHA_VALUES:
    v, w, c = generate_random_mkp_instance(N, M, alpha, seed=INSTANCE_SEED)
    instances[alpha] = {'values': v, 'weights': w, 'capacities': c}
    print(f"  α={alpha}: capacities={c}")

sanity_check(instances)
print("Instances generated and locked.")

  α=0.25: capacities=[632 611 700]
  α=0.5: capacities=[1264 1223 1400]
  α=0.75: capacities=[1896 1835 2100]
  α=0.25: weights shape=(3, 100) ✅
  α=0.5: weights shape=(3, 100) ✅
  α=0.75: weights shape=(3, 100) ✅
Instances generated and locked.


In [ ]:
# ============================================================
# REFERENCE BEST FINDER
# repair-based GA, 100k offspring, 20 runs, take max
# loads from file if already computed — skips recompute
# ============================================================
REF_SAVE_PATH = '/content/reference_bests.json'

def find_reference_best(alpha, n_runs=REF_RUNS, max_offspring=REF_OFFSPRING):
    inst    = instances[alpha]
    v, w, c = inst['values'], inst['weights'], inst['capacities']
    best_overall = 0

    print(f"\nFinding reference best for α={alpha}...")
    for run in range(n_runs):
        r   = run_ga(N, M, v, w, c,
                     population_size=REF_POP,
                     max_offspring=max_offspring,
                     best_known=None,
                     init_method='uniform',
                     seed=run)
        val = r['best_fitness']
        if val > best_overall:
            best_overall = val
        print(f"  Run {run+1:02d}/{n_runs}: best={val:.0f}  "
              f"overall={best_overall:.0f}")

    print(f"  → Reference best for α={alpha}: {best_overall}")
    return int(best_overall)


if os.path.exists(REF_SAVE_PATH):
    # ── Load from file — skip recompute ──────────────────────
    with open(REF_SAVE_PATH, 'r') as f:
        ref_meta = json.load(f)
    reference_bests = {float(k): v for k, v in
                       ref_meta['reference_bests'].items()}
    print("✅ Reference bests loaded from file — skipping recompute:")
    for alpha, ref in reference_bests.items():
        print(f"   α={alpha}: {ref}")

else:
    # ── Compute fresh and save ────────────────────────────────
    reference_bests = {}
    for alpha in ALPHA_VALUES:
        reference_bests[alpha] = find_reference_best(alpha)

    ref_meta = {
        'reference_bests': {str(a): r for a, r in reference_bests.items()},
        'methodology': {
            'description': (
                'Best feasible fitness found by repair-based GA over multiple '
                'runs. Not a proven optimal — a well-resourced best-known value '
                'used as convergence target.'
            ),
            'ga_type':        'repair-based (run_ga)',
            'init_method':    'uniform',
            'population_size': REF_POP,
            'max_offspring':   REF_OFFSPRING,
            'n_runs':          REF_RUNS,
            'seeds':           list(range(REF_RUNS)),
            'selection':       'binary tournament k=2',
            'crossover':       'uniform',
            'mutation_rate':   f'1/n = {1/N:.4f}',
            'instance': {
                'n':              N,
                'm':              M,
                'seed':           INSTANCE_SEED,
                'values_range':   '1-99',
                'weights_range':  '1-49',
                'capacities':     'int(alpha * sum of constraint weights)'
            }
        }
    }

    with open(REF_SAVE_PATH, 'w') as f:
        json.dump(ref_meta, f, indent=2)

    print(f"\nReference bests saved → {REF_SAVE_PATH}")
    print("\nReference bests locked:")
    for alpha, ref in reference_bests.items():
        print(f"   α={alpha}: {ref}")


Finding reference best for α=0.25...
  Run 01/3: best=2375  overall=2375
  Run 02/3: best=2368  overall=2375
  Run 03/3: best=2375  overall=2375
  → Reference best for α=0.25: 2375

Finding reference best for α=0.5...
  Run 01/3: best=3909  overall=3909
  Run 02/3: best=3909  overall=3909
  Run 03/3: best=3909  overall=3909
  → Reference best for α=0.5: 3909

Finding reference best for α=0.75...
  Run 01/3: best=4823  overall=4823
  Run 02/3: best=4823  overall=4823
  Run 03/3: best=4823  overall=4823
  → Reference best for α=0.75: 4823

💾 Reference bests saved → /content/reference_bests.json

✅ Reference bests locked:
   α=0.25: 2375
   α=0.5: 3909
   α=0.75: 4823


In [ ]:
# ============================================================
# p_j COMPUTATION — GPU Monte Carlo, 10M samples
# ============================================================
def compute_pj_gpu(weights, capacities, n_samples=PJ_SAMPLES,
                   batch_size=50_000, seed=PJ_SEED):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Device: {device}")

    m, n  = weights.shape
    W     = torch.tensor(weights,    dtype=torch.float32, device=device)
    C     = torch.tensor(capacities, dtype=torch.float32, device=device)
    torch.manual_seed(seed)

    count_in  = torch.zeros(n, device=device)
    count_out = torch.zeros(n, device=device)
    total_in  = torch.zeros(n, device=device)
    total_out = torch.zeros(n, device=device)

    processed = 0
    t0 = time.time()

    while processed < n_samples:
        bs    = min(batch_size, n_samples - processed)
        X     = torch.randint(0, 2, (bs, n),
                              device=device, dtype=torch.float32)
        usage = X @ W.T  # (bs, m)

        for j in range(n):
            in_j            = X[:, j] == 1
            out_j           = X[:, j] == 0
            usage_without_j = usage - X[:, j:j+1] * W[:, j]
            thresh_in       = C - W[:, j]
            feas_in         = (usage_without_j <= thresh_in).all(dim=1) & in_j
            feas_out        = (usage_without_j <= C        ).all(dim=1) & out_j
            count_in[j]    += feas_in.sum()
            count_out[j]   += feas_out.sum()
            total_in[j]    += in_j.sum()
            total_out[j]   += out_j.sum()

        processed += bs

    rho_in  = (count_in  / total_in.clamp(min=1)).cpu().numpy()
    rho_out = (count_out / total_out.clamp(min=1)).cpu().numpy()
    p_j     = rho_in / (rho_in + rho_out + 1e-12)

    print(f"  Done in {time.time()-t0:.1f}s | "
          f"p_j range [{p_j.min():.3f}, {p_j.max():.3f}] "
          f"mean={p_j.mean():.3f}")
    return p_j

pj_values = {}
for alpha in ALPHA_VALUES:
    inst = instances[alpha]
    print(f"\nComputing p_j for α={alpha}...")
    pj_values[alpha] = compute_pj_gpu(
        inst['weights'], inst['capacities']
    )

print("\n✅ p_j computed for all tightness levels.")


Computing p_j for α=0.25...
  Device: cuda
  Done in 7.3s | p_j range [0.000, 1.000] mean=0.240

Computing p_j for α=0.5...
  Device: cuda
  Done in 6.4s | p_j range [0.414, 0.493] mean=0.449

Computing p_j for α=0.75...
  Device: cuda
  Done in 7.1s | p_j range [0.500, 0.500] mean=0.500

✅ p_j computed for all tightness levels.


In [ ]:
# ============================================================
# EXPERIMENT PIPELINE — function definition
# ============================================================
def make_serializable(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_serializable(i) for i in obj]
    return obj

def run_experiment(label, alpha, penalty_type, n_runs=N_RUNS):
    inst     = instances[alpha]
    v, w, c  = inst['values'], inst['weights'], inst['capacities']
    ref      = reference_bests[alpha]
    pj       = pj_values[alpha]
    ga_fn    = (run_ga_death_penalty if penalty_type == 'death'
                else run_ga_penalty_fitness)

    print(f"\n{'='*65}")
    print(f"  EXPERIMENT: {label}")
    print(f"{'='*65}")
    print(f"  Instance       : n={N}, m={M}, seed={INSTANCE_SEED}")
    print(f"  Tightness      : α={alpha}")
    print(f"  Penalty        : {penalty_type}" +
          (f"  (multiplier={PENALTY_MULT})" if penalty_type != 'death' else ""))
    print(f"  Runs           : {n_runs}")
    print(f"  Offspring/run  : {N_OFFSPRING}")
    print(f"  Population     : {POP_SIZE}")
    print(f"  Mutation rate  : 1/n = {1/N:.4f}")
    print(f"  Tournament     : binary (k=2)")
    print(f"  Crossover      : uniform")
    print(f"  Reference best : {ref}")
    print(f"  p_j samples    : {PJ_SAMPLES:,}")
    print(f"  Thresholds     : {CONVERGENCE_THRESHOLDS}")
    print(f"{'='*65}")

    results_uni = []
    results_bia = []

    for run in range(n_runs):
        print(f"  Run {run+1:02d}/{n_runs}...", end=" ", flush=True)

        base_kwargs = dict(
            n=N, m=M,
            values=v, weights=w, capacities=c,
            population_size=POP_SIZE,
            max_offspring=N_OFFSPRING,
            best_known=ref,
            track_every=TRACK_EVERY,
        )
        if penalty_type != 'death':
            base_kwargs['penalty_multiplier'] = PENALTY_MULT

        r_uni = ga_fn(**base_kwargs,
                      init_method='uniform',
                      p_j_values=None,
                      seed=run * 7)

        r_bia = ga_fn(**{**base_kwargs, 'seed': run * 7 + 1000},
                      init_method='biased',
                      p_j_values=pj)

        results_uni.append(r_uni)
        results_bia.append(r_bia)

        print(f"U feas0={r_uni['feasibility_rate_gen0']:.1%} "
              f"B feas0={r_bia['feasibility_rate_gen0']:.1%} "
              f"U final={r_uni['pct_of_best_final']:.1f}% "
              f"B final={r_bia['pct_of_best_final']:.1f}%")

    save_key  = label.replace(" ","_").replace("=","").replace(".","")
    save_path = f'/content/results_{save_key}.json'
    with open(save_path, 'w') as f:
        json.dump({
            'label':   label,
            'alpha':   alpha,
            'penalty': penalty_type,
            'ref':     ref,
            'params': {
                'n': N, 'm': M, 'instance_seed': INSTANCE_SEED,
                'pop_size': POP_SIZE, 'n_runs': n_runs,
                'n_offspring': N_OFFSPRING,
                'mutation_rate': 1/N,
                'penalty_mult': PENALTY_MULT,
                'pj_samples': PJ_SAMPLES,
                'pj_seed': PJ_SEED,
                'ref_offspring': REF_OFFSPRING,
                'ref_runs': REF_RUNS,
                'tournament': 'binary k=2',
                'crossover': 'uniform',
                'convergence_thresholds': CONVERGENCE_THRESHOLDS,
            },
            'uniform': [make_serializable(
                            {k: v for k, v in r.items() if k != 'curve'})
                        for r in results_uni],
            'biased':  [make_serializable(
                            {k: v for k, v in r.items() if k != 'curve'})
                        for r in results_bia],
        }, f, indent=2)
    print(f"\n  💾 Saved → {save_path}")

    summarize_comparison(results_uni, results_bia, label, ref)
    return results_uni, results_bia

print("run_experiment defined.")

run_experiment defined.


In [ ]:
# ============================================================
# RUN ALL 6 CONDITIONS
# ============================================================
all_results = {}

conditions = [
    ("Death penalty α=0.25", 0.25, "death"),
    ("Death penalty α=0.50", 0.50, "death"),
    ("Death penalty α=0.75", 0.75, "death"),
    ("Penalty x5.0  α=0.25", 0.25, "penalty"),
    ("Penalty x5.0  α=0.50", 0.50, "penalty"),
    ("Penalty x5.0  α=0.75", 0.75, "penalty"),
]

for label, alpha, penalty_type in conditions:
    uni, bia = run_experiment(label, alpha, penalty_type)
    all_results[label] = (uni, bia)

print("\n✅ ALL BASELINE EXPERIMENTS COMPLETE")


  EXPERIMENT: Death penalty α=0.25
  Instance       : n=100, m=3, seed=42
  Tightness      : α=0.25
  Penalty        : death
  Runs           : 20
  Offspring/run  : 20000
  Population     : 50
  Mutation rate  : 1/n = 0.0100
  Tournament     : binary (k=2)
  Crossover      : uniform
  Reference best : 2375
  p_j samples    : 10,000,000
  Thresholds     : [50, 60, 70, 80, 90, 95, 98, 99]
  Run 01/20... U feas0=0.0% B feas0=42.0% U final=0.0% B final=95.9%
  Run 02/20... U feas0=0.0% B feas0=52.0% U final=0.0% B final=96.5%
  Run 03/20... U feas0=0.0% B feas0=38.0% U final=0.0% B final=97.9%
  Run 04/20... U feas0=0.0% B feas0=40.0% U final=0.0% B final=98.0%
  Run 05/20... U feas0=0.0% B feas0=50.0% U final=0.0% B final=95.3%
  Run 06/20... U feas0=0.0% B feas0=48.0% U final=0.0% B final=96.5%
  Run 07/20... U feas0=0.0% B feas0=56.0% U final=0.0% B final=97.4%
  Run 08/20... U feas0=0.0% B feas0=46.0% U final=0.0% B final=95.3%
  Run 09/20... U feas0=0.0% B feas0=54.0% U final=0.0% B

In [ ]:
import json
from datetime import date

CONVERGENCE_THRESHOLDS = [50, 60, 70, 80, 90, 95, 98, 99]
N, M, INSTANCE_SEED    = 100, 3, 42
POP_SIZE               = 50
N_RUNS                 = 20
N_OFFSPRING            = 20000
PJ_SAMPLES             = 10_000_000
PJ_SEED                = 42
REF_OFFSPRING          = 30_000
REF_RUNS               = 20
PENALTY_MULT           = 5.0

with open('reference_bests.json') as f:
    ref_meta = json.load(f)
reference_bests = {float(k): v for k, v in
                   ref_meta['reference_bests'].items()}

conditions = [
    ('results_Death_penalty_α025.json', 'Death penalty a=0.25'),
    ('results_Death_penalty_α050.json', 'Death penalty a=0.50'),
    ('results_Death_penalty_α075.json', 'Death penalty a=0.75'),
    ('results_Penalty_x50__α025.json',  'Penalty x5.0 a=0.25'),
    ('results_Penalty_x50__α050.json',  'Penalty x5.0 a=0.50'),
    ('results_Penalty_x50__α075.json',  'Penalty x5.0 a=0.75'),
]

def mean_field(runs, field):
    vals = [r[field] for r in runs if r.get(field) is not None]
    return sum(vals)/len(vals) if vals else 0.0

def thresh_stats(runs, t):
    vals = []
    for r in runs:
        c = r.get('convergence', {})
        v = c.get(str(t)) if c.get(str(t)) is not None else c.get(t)
        if v is not None:
            vals.append(v)
    if not vals:
        return 'never', 0
    return f"{sum(vals)/len(vals):.0f}", len(vals)

lines = []
lines.append("# Baseline Experiment Results")
lines.append(f"\nGenerated: {date.today()}")
lines.append("\n## Experiment Parameters\n")
lines.append("| Parameter | Value |")
lines.append("|---|---|")
lines.append(f"| n (items) | {N} |")
lines.append(f"| m (constraints) | {M} |")
lines.append(f"| Instance seed | {INSTANCE_SEED} |")
lines.append(f"| Population size | {POP_SIZE} |")
lines.append(f"| Runs per condition | {N_RUNS} |")
lines.append(f"| Offspring per run | {N_OFFSPRING} |")
lines.append(f"| Mutation rate | 1/n = {1/N:.4f} |")
lines.append(f"| Tournament | binary k=2 |")
lines.append(f"| Crossover | uniform |")
lines.append(f"| p_j samples | {PJ_SAMPLES:,} |")
lines.append(f"| Convergence thresholds | {CONVERGENCE_THRESHOLDS} |")

lines.append("\n## Reference Bests\n")
lines.append(
    f"Established via repair-based GA, {REF_RUNS} runs, "
    f"{REF_OFFSPRING:,} offspring, seeds 0-{REF_RUNS-1}. "
    "Not proven optimal -- best known values used as convergence targets.\n"
)
lines.append("| Alpha | Reference Best |")
lines.append("|---|---|")
for alpha, ref in sorted(reference_bests.items()):
    lines.append(f"| {alpha} | {ref} |")

lines.append("\n## p_j Summary\n")
lines.append("| Alpha | p_j min | p_j max | p_j mean | Interpretation |")
lines.append("|---|---|---|---|---|")
lines.append("| 0.25 | 0.000 | 1.000 | 0.240 | Strong differentiation -- tight instance |")
lines.append("| 0.50 | 0.414 | 0.493 | 0.449 | Weak differentiation -- medium instance |")
lines.append("| 0.75 | 0.500 | 0.500 | 0.500 | No differentiation -- degenerates to uniform |")

lines.append("\n## Results by Condition\n")

for fname, label in conditions:
    with open(fname) as f:
        d = json.load(f)

    ref  = d['ref']
    uni  = d['uniform']
    bia  = d['biased']
    n_runs = len(uni)

    lines.append(f"### {label}\n")
    lines.append(f"Reference best: {ref}  |  Runs: {n_runs}  "
                 f"|  Offspring: {d['params']['n_offspring']}\n")
    lines.append("| Metric | Uniform | Biased |")
    lines.append("|---|---|---|")
    lines.append(
        f"| Feasibility at gen 0 "
        f"| {mean_field(uni,'feasibility_rate_gen0'):.1%} "
        f"| {mean_field(bia,'feasibility_rate_gen0'):.1%} |"
    )
    lines.append(
        f"| Fitness at gen 0 (% of ref) "
        f"| {mean_field(uni,'pct_of_best_gen0'):.1f}% "
        f"| {mean_field(bia,'pct_of_best_gen0'):.1f}% |"
    )
    lines.append(
        f"| Final fitness (% of ref) "
        f"| {mean_field(uni,'pct_of_best_final'):.1f}% "
        f"| {mean_field(bia,'pct_of_best_final'):.1f}% |"
    )
    for t in CONVERGENCE_THRESHOLDS:
        u_mean, u_count = thresh_stats(uni, t)
        b_mean, b_count = thresh_stats(bia, t)
        u_str = f"{u_mean} ({u_count}/{n_runs})" if u_mean != 'never' else 'never'
        b_str = f"{b_mean} ({b_count}/{n_runs})" if b_mean != 'never' else 'never'
        lines.append(f"| Offspring to {t}% | {u_str} | {b_str} |")

    lines.append("")

md = '\n'.join(lines)
with open('RESULTS.md', 'w') as f:
    f.write(md)
print("RESULTS.md written")

from google.colab import files
files.download('RESULTS.md')

RESULTS.md written


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>